# L13: NLP基础与文本表示


# L13: NLP基础与文本表示

**课时**: 2 课时（90 分钟/课时）

## 1. 学习目标

完成本课程后，学员能够：
1. 理解自然语言处理（NLP）的基本任务：分类、序列标注、生成
2. 掌握文本预处理方法：分词、标准化、去停用词
3. 解释词袋模型（BoW）、TF-IDF、词嵌入的原理与局限
4. 使用NLTK/spaCy进行文本预处理和分析
5. 理解词向量的语义特性：相似度、类比推理

## 2. 核心知识点

### 2.1 NLP概述与任务分类

**NLP基本任务**:
| 任务类型 | 输入 | 输出 | 典型应用 |
|----------|------|------|----------|
| 文本分类 | 文本 | 类别标签 | 垃圾邮件检测、情感分析 |
| 序列标注 | 文本 | 每个词的标签 | 命名实体识别、词性标注 |
| 文本生成 | 文本/条件 | 文本 | 机器翻译、摘要生成 |
| 问答系统 | 问题+上下文 | 答案 | 客服机器人、信息检索 |
| 对话系统 | 对话历史+输入 | 回复 | 聊天机器人 |

**NLP发展历程**:
- 规则阶段（1950s-1990s）：基于语言学规则
- 统计学习（1990s-2012）：词袋模型、SVM、HMM
- 深度学习（2012-2017）：Word2Vec、CNN/RNN for NLP
- 预训练时代（2018-至今）：BERT、GPT、T5

### 2.2 文本预处理


In [ ]:
import nltk
import spacy

# 下载必要资源
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')

# 加载英文模型
nlp = spacy.load("en_core_web_sm")

text = "Natural Language Processing (NLP) is a fascinating field of AI. Stanford University researchers developed many key algorithms."

# ============ 1. 分词 (Tokenization) ============
from nltk.tokenize import word_tokenize

tokens = word_tokenize(text.lower())
print(f"分词结果: {tokens}")

# spaCy分词（保留词性信息）
doc = nlp(text)
print("\nspaCy分词:")
for token in doc:
    print(f"  {token.text:15} | POS: {token.pos_:5} | Lemma: {token.lemma_}")

# ============ 2. 停用词去除 ============
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
filtered_tokens = [w for w in tokens if w not in stop_words and w.isalpha()]
print(f"\n去停用词后: {filtered_tokens}")

# ============ 3. 词性标注 (POS Tagging) ============
pos_tags = nltk.pos_tag(tokens)
print(f"\n词性标注: {pos_tags}")

# ============ 4. 命名实体识别 (NER) ============
print("\n命名实体:")
for ent in doc.ents:
    print(f"  {ent.text:15} -> {ent.label_}")

# ============ 5. 词干提取 vs 词形还原 ============
from nltk.stem import PorterStemmer, WordNetLemmatizer

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

words = ["running", "ran", "runs", "better", "studies"]
print("\n词干提取 vs 词形还原:")
for w in words:
    print(f"  {w:10} | Stem: {stemmer.stem(w):10} | Lemma: {lemmatizer.lemmatize(w, pos='v')}")


### 2.3 文本表示方法

**词袋模型 (Bag of Words)**:


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

corpus = [
    "I love machine learning",
    "Machine learning is amazing",
    "Deep learning is part of AI",
    "I prefer machine learning over deep learning"
]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus)

print("词汇表:", vectorizer.get_feature_names_out())
print("\n词袋矩阵:")
print(X.toarray())


**TF-IDF (Term Frequency-Inverse Document Frequency)**:


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(corpus)

print("TF-IDF矩阵:")
print(X_tfidf.toarray())

# 手动计算TF-IDF
import numpy as np

def compute_tfidf(doc, corpus):
    # TF: 词在文档中出现的次数
    # IDF: log(总文档数 / 包含该词的文档数)
    tf = doc.count(word) / len(doc.split())
    idf = np.log(len(corpus) / sum(1 for d in corpus if word in d))
    return tf * idf

# 示例
word = "machine"
doc_idx = 0
tfidf_value = X_tfidf[doc_idx, tfidf.get_feature_names_out().tolist().index(word)]
print(f"\n'{word}'在文档0中的TF-IDF: {tfidf_value:.4f}")


**词嵌入 (Word Embedding)**:


In [ ]:
# 使用预训练词向量（GloVe）
import numpy as np

# 加载GloVe（需要下载glove.6B.100d.txt）
def load_glove(path):
    embeddings = {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            vector = np.array(values[1:], dtype='float32')
            embeddings[word] = vector
    return embeddings

# 演示：计算词相似度
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# 加载预训练向量后
# embedding = load_glove('glove.6B.100d.txt')
# similarity = cosine_similarity(embedding['king'], embedding['queen'])
# print(f"king-queen相似度: {similarity:.3f}")

# 使用gensim加载预训练模型
from gensim.models import KeyedVectors

# 演示：词向量类比
print("词向量类比演示:")
print("king - man + woman ≈ ?")
print("答案：queen（通过向量运算得到）")


### 2.4 词向量的语义特性


In [ ]:
"""
L13-word-embeddings.py
演示词向量的语义特性：相似度、类比推理
"""

from gensim.models import Word2Vec
import numpy as np

# 训练词向量模型（演示用）
sentences = [
    ['king', 'was', 'a', 'man'],
    ['queen', 'was', 'a', 'woman'],
    ['man', 'walked', 'on', 'street'],
    ['woman', 'walked', 'on', 'street'],
    ['king', 'ruled', 'kingdom'],
    ['queen', 'ruled', 'kingdom'],
]

# 训练Word2Vec（实际应用中用大规模语料）
model = Word2Vec(sentences, vector_size=10, window=2, min_count=1, epochs=100)

# 获取词向量
king_vec = model.wv['king']
queen_vec = model.wv['queen']
man_vec = model.wv['man']
woman_vec = model.wv['woman']

print("词向量维度:", king_vec.shape)

# 计算相似度
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"\nking-queen相似度: {cosine_sim(king_vec, queen_vec):.3f}")
print(f"king-man相似度: {cosine_sim(king_vec, man_vec):.3f}")

# 类比推理：king - man + woman ≈ queen
analogy_vec = king_vec - man_vec + woman_vec
similar_words = model.wv.similar_by_vector(analogy_vec, topn=3)
print(f"\nking - man + woman = {similar_words[0][0]}")


## 3. 代码示例


In [ ]:
"""
L13-nlp-preprocessing.py
完整NLP预处理流程：文本清洗→分词→向量化
"""

import re
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# 示例文本
texts = [
    "I love machine learning! It's absolutely amazing. #AI #ML",
    "Deep learning models are getting better every day.",
    "Natural Language Processing (NLP) helps computers understand human language.",
    "Machine learning is a subset of artificial intelligence.",
    "Transformers revolutionized deep learning in 2017."
]

def preprocess_text(text):
    """文本预处理流程"""
    # 1. 转小写
    text = text.lower()
    
    # 2. 去除URL
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # 3. 去除特殊字符（保留字母和空格）
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # 4. 分词
    tokens = text.split()
    
    # 5. 去停用词
    stop_words = set(stopwords.words('english'))
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    
    return ' '.join(tokens)

# 预处理
processed = [preprocess_text(t) for t in texts]
print("预处理结果:")
for i, (orig, proc) in enumerate(zip(texts, processed)):
    print(f"  [{i+1}] {proc}")

# TF-IDF向量化
tfidf = TfidfVectorizer(max_features=20, ngram_range=(1, 2))
X = tfidf.fit_transform(processed)

print(f"\n词汇表: {tfidf.get_feature_names_out()}")
print(f"TF-IDF矩阵形状: {X.shape}")

# 显示TF-IDF值
print("\nTF-IDF矩阵（部分）:")
feature_names = tfidf.get_feature_names_out()
for i, doc in enumerate(processed):
    print(f"文档{i+1}:")
    scores = X[i].toarray()[0]
    for j, score in enumerate(scores):
        if score > 0:
            print(f"  {feature_names[j]}: {score:.3f}")


## 4. 练习题

1. **分词对比**: 使用NLTK和spaCy对同一段中文文本进行分词，比较两者在处理歧义、专有名词、成语时的差异。
2. **向量化实验**: 对一组文本（可从新闻网站爬取），分别使用BoW、TF-IDF、预训练词向量进行表示，在分类任务上比较三者的效果。
3. **词向量类比**: 加载预训练词向量（如GloVe或Word2Vec），实现以下类比推理：Paris - France + Italy = ?、king - man + woman = ?
4. **文本清洗**: 设计一个文本清洗Pipeline，处理以下情况：去除HTML标签、表情符号、重复空格、特殊编码，并比较清洗前后的NLP任务效果。
5. **停用词分析**: 分析停用词对不同NLP任务的影响——文本分类、情感分析、关键词提取——设计自适应停用词策略。

## 5. 延伸阅读

- **书籍**: Jurafsky & Martin, *Speech and Language Processing*（第3版）— NLP领域权威教材
- **网站**: NLTK官方文档 — https://www.nltk.org/
- **网站**: spaCy官方文档 — https://spacy.io/
- **资源**: Gensim — 词向量训练工具 https://radimrehurek.com/gensim/
- **资源**: HuggingFace Tokenizers — 快速分词工具 https://huggingface.co/docs/tokenizers/

---
